In [ ]:
# ================================================================
# [1/12]  SETUP & IMPORTS
# ================================================================

import os, shutil, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print("✅ Imports completed")

# ================================================================
# [2/12]  GPU VERIFICATION & MIXED PRECISION
# ================================================================

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("✅ GPU detected:", gpus)
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("✅ Mixed precision enabled for faster training")
    except Exception as e:
        print("⚠️ GPU setup warning:", e)
else:
    print("⚠️ No GPU detected. Training will be slower on CPU.")

# ================================================================
# [3/12]  LOAD LABEL FILE (Excel or CSV)
# ================================================================

# Upload your Excel or CSV file to /content/
LABEL_FILE = '/content/ozone_2017_2025_monthly_label.xlsx'  # change to .csv if needed
IMAGES_PATH = '/content/uploaded_images/'                   # all images here

if LABEL_FILE.endswith('.xlsx'):
    df = pd.read_excel(LABEL_FILE)
else:
    df = pd.read_csv(LABEL_FILE)

print("✅ Label file loaded with shape:", df.shape)
print(df.head())

# ================================================================
# [4/12]  VALIDATION CHECKS
# ================================================================

required_cols = {'filename', 'label'}
if not required_cols.issubset(df.columns):
    raise ValueError(f"❌ Missing required columns in label file. Found: {df.columns}")

# Filter only available images
available_files = {f for f in os.listdir(IMAGES_PATH) if f.lower().endswith('.png')}
df = df[df['filename'].isin(available_files)]
print("✅ Matched images with labels:", len(df))

# Class distribution
print("Class distribution:\n", df['label'].value_counts())

# ================================================================
# [5/12]  TRAIN/VAL/TEST SPLIT
# ================================================================

train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
val_df, test_df  = train_test_split(temp_df, test_size=0.33, stratify=temp_df['label'], random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# ================================================================
# [6/12]  IMAGE DATA GENERATORS + AUGMENTATION
# ================================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=30,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    validation_split=None
)

test_val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=IMAGES_PATH,
    x_col='filename',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_gen = test_val_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=IMAGES_PATH,
    x_col='filename',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = test_val_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=IMAGES_PATH,
    x_col='filename',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("✅ Data generators ready")

# ================================================================
# [7/12]  BUILD TRANSFER LEARNING MODEL (VGG16)
# ================================================================

base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # freeze convolutional base

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(2, activation='softmax', dtype='float32')
])

LEARNING_RATE = 1e-4
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# ================================================================
# [8/12]  CALLBACKS
# ================================================================

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_ozone_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

# ================================================================
# [9/12]  TRAIN MODEL
# ================================================================

EPOCHS = 50

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# ================================================================
# [10/12]  EVALUATE ON TEST SET
# ================================================================

test_loss, test_acc = model.evaluate(test_gen)
print(f"\n✅ Test Accuracy: {test_acc:.3f} | Test Loss: {test_loss:.3f}")

# ================================================================
# [11/12]  CONFUSION MATRIX & REPORT
# ================================================================

preds = model.predict(test_gen)
pred_labels = np.argmax(preds, axis=1)
true_labels = test_gen.classes
class_names = list(test_gen.class_indices.keys())

cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png')
plt.show()

print("\nClassification Report:")
print(classification_report(true_labels, pred_labels, target_names=class_names))

# ================================================================
# [12/12]  VISUALIZE TRAINING HISTORY + SAVE MODEL
# ================================================================

plt.figure(figsize=(10,4))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Accuracy')
plt.legend()

# Loss plot
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.legend()

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

# Save final model
model.save('ozone_vgg16_final.h5')
model.save('ozone_vgg16_savedmodel/')
print("✅ Models saved: ozone_vgg16_final.h5 and ozone_vgg16_savedmodel/")

# ================================================================
#  OPTIONAL: SAMPLE INFERENCE ON ONE TEST IMAGE
# ================================================================

import random
sample_path = os.path.join(IMAGES_PATH, random.choice(test_df['filename'].tolist()))
img = tf.keras.preprocessing.image.load_img(sample_path, target_size=IMG_SIZE)
img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
img_batch = np.expand_dims(img_array, axis=0)

pred = model.predict(img_batch)
pred_class = class_names[np.argmax(pred)]
confidence = np.max(pred) * 100

plt.imshow(img)
plt.title(f"True: {test_df[test_df['filename']==os.path.basename(sample_path)]['label'].values[0]} | Pred: {pred_class} ({confidence:.1f}%)")
plt.axis('off')
plt.show()


✅ Imports completed
⚠️ No GPU detected. Training will be slower on CPU.
✅ Label file loaded with shape: (106, 7)
         filename  year  month  day  ozone_hole_area_million_km2  \
0  2017-01-01.png  2017      1    1                         17.4   
1  2017-02-01.png  2017      2    1                         17.4   
2  2017-03-01.png  2017      3    1                         17.4   
3  2017-04-01.png  2017      4    1                         17.4   
4  2017-05-01.png  2017      5    1                         17.4   

   minimum_ozone_DU     label  
0             141.8  depleted  
1             141.8  depleted  
2             141.8  depleted  
3             141.8  depleted  
4             141.8  depleted  
✅ Matched images with labels: 106
Class distribution:
 label
depleted    94
normal      12
Name: count, dtype: int64
Train: 74, Val: 21, Test: 11
Found 74 validated image filenames belonging to 2 classes.
Found 21 validated image filenames belonging to 2 classes.
Found 11 validated ima

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,780,610 (56.38 MB)

 Trainable params: 65,922 (257.51 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11s/step - accuracy: 0.3228 - loss: 1.3732 
Epoch 1: val_accuracy improved from -inf to 0.14286, saving model to best_ozone_model.h5


5/5 ━━━━━━━━━━━━━━━━━━━━ 77s 15s/step - accuracy: 0.3253 - loss: 1.3639 - val_accuracy: 0.1429 - val_loss: 0.8705 - learning_rate: 1.0000e-04
Epoch 2/50
1/5 ━━━━━━━━━━━━━━━━━━━━ 59s 15s/step - accuracy: 0.5000 - loss: 0.9764

KeyboardInterrupt: 

In [ ]:
# ============================================================
# OZONE DEPLETION DETECTION USING CNN (VGG16 PRETRAINED)
# ============================================================

# STEP 1: IMPORT LIBRARIES
# ------------------------------------------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# ============================================================
# STEP 2: LOAD THE DATASET FROM EXCEL
# ------------------------------------------------------------
# Excel structure expected:
# | filename | label | ... (other columns)
# label → "normal" or "depleted"

df = pd.read_excel("/content/ozone_2017_2025_monthly_label.xlsx")  # Adjust path if needed

print("✅ Dataset loaded successfully.")
print(df.head())
print(df['label'].value_counts())

# ============================================================
# STEP 3: PREPARE IMAGE AND LABEL ARRAYS
# ------------------------------------------------------------
IMG_SIZE = (224, 224)
IMAGES_PATH = '/content/uploaded_images/' # Define IMAGES_PATH here

images = []
labels = []

for i, row in df.iterrows():
    # Use 'filename' column and join with IMAGES_PATH
    img_fname = row['filename']
    img_path = os.path.join(IMAGES_PATH, img_fname)
    label = row['label']
    if os.path.exists(img_path):
        img = load_img(img_path, target_size=IMG_SIZE)
        img_array = img_to_array(img) / 255.0
        images.append(img_array)
        labels.append(0 if label.lower() == 'normal' else 1)  # 0 = normal, 1 = depleted
    else:
        print(f"⚠️ Image not found: {img_path}. Skipping.")


images = np.array(images)
labels = np.array(labels)

print(f"\nTotal images loaded: {len(images)}")
print(f"Image shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

# ============================================================
# STEP 4: TRAIN-TEST SPLIT
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

# ============================================================
# STEP 5: DEFINE THE PRETRAINED MODEL (VGG16)
# ------------------------------------------------------------
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom classifier head
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ============================================================
# STEP 6: HANDLE CLASS IMBALANCE WITH CLASS WEIGHTS
# ------------------------------------------------------------
print("\n[INFO] Computing class weights for imbalance...")

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

print(f"  Class weights: {class_weight_dict}")
print(f"  Model will naturally learn from imbalance without oversampling/undersampling.")

# ============================================================
# STEP 7: TRAIN THE MODEL
# ------------------------------------------------------------
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=16,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# STEP 8: EVALUATE THE MODEL
# ------------------------------------------------------------
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\n✅ Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Depleted']))

acc = accuracy_score(y_test, y_pred)
print(f"Overall Model Accuracy: {acc * 100:.2f}%")

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# ============================================================
# STEP 9: VISUALIZE TRAINING PERFORMANCE
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# ============================================================
# STEP 10: TEST SOME RANDOM PREDICTIONS
# ------------------------------------------------------------
print("\nSample Predictions:")
# Need to map predicted label index back to class name
label_map = {0: 'Normal', 1: 'Depleted'}
for i in range(5):
    idx = np.random.randint(0, len(X_test))
    img = X_test[idx]
    pred_label_idx = np.argmax(model.predict(np.expand_dims(img, axis=0)), axis=1)[0]
    pred_label_name = label_map[pred_label_idx]
    actual = y_test[idx]
    actual_label_name = label_map[actual]
    print(f"Image {i+1}: Predicted = {pred_label_name} | Actual = {actual_label_name}")

In [ ]:
"""
Ozone Layer Depletion Detection
Complete Google Colab-ready Python script (copy-paste into a Colab cell and run).
Sections: [1/12] to [12/12] as specified.

Notes:
- Upload your 108 PNG images to /content/uploaded_images/ using Colab Files panel.
- Upload ozone_2017_2025_monthly_label.csv to /content/ using Colab Files panel.
- This script organizes files, builds a VGG16 transfer learning model, trains with augmentation,
  evaluates on a test set, saves model and plots, and demonstrates a sample inference.
"""

# [1/12] Setup & Imports
print("[1/12] Setup & Imports - Installing and importing libraries...")

try:
    # Colab already has these, but ensure up-to-date TensorFlow
    import os
    import sys
    import shutil
    import math
    import random
    import time
    from pathlib import Path
    import warnings

    warnings.filterwarnings("ignore")

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from tensorflow.keras.models import Model
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

    from sklearn.model_selection import train_test_split
    from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

    print("✓ Libraries imported")
except Exception as e:
    print("ERROR importing libraries:", e)
    raise

# [2/12] GPU Verification: Check T4 GPU, enable mixed precision
print("\n[2/12] GPU Verification & Mixed Precision Setup")

try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            # Enable memory growth for GPUs to avoid OOM
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            gpu_details = tf.config.experimental.get_device_details(gpus[0])
            print(f"✓ GPU detected: {gpus[0].name if hasattr(gpus[0], 'name') else gpus[0]}")
        except Exception as e:
            print("Warning: Could not set memory growth:", e)
    else:
        print("Warning: No GPU detected. Training will proceed on CPU (slower).")

    # Enable mixed precision for faster training on supported GPUs (T4)
    try:
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy('mixed_float16')
        print("✓ Mixed precision enabled (policy='mixed_float16')")
    except Exception as e:
        print("Warning: Mixed precision not enabled:", e)

except Exception as e:
    print("ERROR during GPU verification:", e)
    raise

# [3/12] Load CSV Labels: Read labeling CSV, create filename→label map
print("\n[3/12] Load CSV Labels - Reading CSV and validating...")

# Corrected file path to use the Excel file
CSV_PATH = "/content/ozone_2017_2025_monthly_label.xlsx"
uploaded_images_dir = "/content/uploaded_images"
organized_base_dir = "/content/ozone_dataset"

try:
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(
            f"Data file not found at {CSV_PATH}. Please upload ozone_2017_2025_monthly_label.xlsx to /content/."
        )

    # Use pd.read_excel to load the data
    df_labels = pd.read_excel(CSV_PATH)
    required_cols = {'filename', 'year', 'month', 'day', 'ozone_hole_area_million_km2', 'minimum_ozone_DU', 'label'}
    if not required_cols.issubset(set(df_labels.columns)):
        raise ValueError(f"Data file missing columns. Required columns: {required_cols}")

    # Normalize filename strings (strip)
    df_labels['filename'] = df_labels['filename'].astype(str).str.strip()

    # Validate labels and classification criteria (recompute label optionally)
    def classify_row(row):
        try:
            area = float(row['ozone_hole_area_million_km2'])
            min_oz = float(row['minimum_ozone_DU'])
            return 'depleted' if (area > 15.0 or min_oz < 130.0) else 'normal'
        except:
            return row.get('label', 'normal')

    # Recompute label column based on supplied criteria to ensure consistency
    df_labels['label_computed'] = df_labels.apply(classify_row, axis=1)
    # If original label exists but disagrees, prefer computed label and warn.
    mismatches = (df_labels['label'].astype(str).str.lower() != df_labels['label_computed'])
    if mismatches.any():
        print(f"Warning: {mismatches.sum()} data file labels differ from computed labels. Using computed labels.")
    df_labels['label'] = df_labels['label_computed']
    df_labels = df_labels.drop(columns=['label_computed'])
    filename_to_label = dict(zip(df_labels['filename'], df_labels['label']))

    print(f"✓ Data file loaded: {len(df_labels)} rows")
except Exception as e:
    print("ERROR loading data file:", e)
    raise

# [4/12] Organize Images: Scan uploaded images, match with labels, organize folders
print("\n[4/12] Organize Images - Scanning uploaded images and organizing into train/val/test folders")

# Create dataset directories
train_dir = os.path.join(organized_base_dir, "train")
val_dir = os.path.join(organized_base_dir, "validation")
test_dir = os.path.join(organized_base_dir, "test")
for base in [train_dir, val_dir, test_dir]:
    for cls in ['normal', 'depleted']:
        os.makedirs(os.path.join(base, cls), exist_ok=True)

# Validate uploaded_images_dir
try:
    if not os.path.exists(uploaded_images_dir):
        raise FileNotFoundError(f"Uploaded images folder not found at {uploaded_images_dir}. Please create and upload files.")

    # Get all png files in uploaded_images_dir (non-recursive)
    image_files = [f for f in os.listdir(uploaded_images_dir) if f.lower().endswith('.png')]
    if len(image_files) == 0:
        raise FileNotFoundError(f"No PNG images found in {uploaded_images_dir}. Please upload 108 PNG images.")

    print(f"Found {len(image_files)} PNG files in {uploaded_images_dir}")

    # Match images to CSV labels, build list of (filename, label, src_path)
    matched = []
    skipped_no_label = []
    for fname in image_files:
        if fname in filename_to_label:
            label = filename_to_label[fname].lower()
            if label not in ['normal', 'depleted']:
                print(f"Warning: Unknown label '{label}' in data file for {fname}. Skipping.")
                skipped_no_label.append(fname)
                continue
            matched.append((fname, label, os.path.join(uploaded_images_dir, fname)))
        else:
            skipped_no_label.append(fname)

    if len(matched) == 0:
        raise ValueError("No uploaded images matched data file filenames. Check filename formats (yyyy-mm-dd.png).")

    print(f"✓ {len(matched)} images matched to labels; {len(skipped_no_label)} skipped due to missing data file entries")
    if skipped_no_label:
        print("Skipped filenames (not in data file):", skipped_no_label)

except Exception as e:
    print("ERROR organizing images:", e)
    raise

# [5/12] Train/Val/Test Split: Split data with stratification (70/20/10)
print("\n[5/12] Train/Val/Test Split - Stratified split (70%/20%/10%)")

try:
    # Build DataFrame for splitting
    df_images = pd.DataFrame(matched, columns=['filename', 'label', 'src_path'])
    class_counts = df_images['label'].value_counts().to_dict()
    print("Class distribution before split:", class_counts)

    # Basic validation checks
    if len(df_images) < 20:
        raise ValueError("Insufficient images (<20) for training. Need at least 20 images.")

    for cls, cnt in class_counts.items():
        if cnt < 5:
            raise ValueError(f"Insufficient images for class '{cls}' ({cnt} images). Need at least 5 per class.")

    # First split train vs temp (train 70%)
    train_df, temp_df = train_test_split(
        df_images,
        stratify=df_images['label'],
        test_size=0.30,
        random_state=42
    )

    # Then split temp into val (20%) and test (10%) relative to total images:
    # temp proportion = 30% total; of that, validation should be (20/30)=66.666...% of temp,
    # test should be (10/30)=33.333...% of temp.
    val_fraction_of_temp = 20.0 / 30.0
    val_df, test_df = train_test_split(
        temp_df,
        stratify=temp_df['label'],
        test_size=(1 - val_fraction_of_temp),
        random_state=42
    )

    print(f"Split sizes -> Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")

    # Copy files into organized folder structure using shutil.copy2 to preserve metadata
    def safe_copy_rows(df_rows, dest_base):
        copied = 0
        for _, row in df_rows.iterrows():
            src = row['src_path']
            dst_dir = os.path.join(dest_base, row['label'])
            dst = os.path.join(dst_dir, row['filename'])
            try:
                shutil.copy2(src, dst)
                copied += 1
            except Exception as e:
                print(f"Warning: Failed to copy {src} -> {dst}: {e}")
        return copied

    copied_train = safe_copy_rows(train_df, train_dir)
    copied_val = safe_copy_rows(val_df, val_dir)
    copied_test = safe_copy_rows(test_df, test_dir)

    print(f"Files copied -> Train: {copied_train}, Validation: {copied_val}, Test: {copied_test}")

except Exception as e:
    print("ERROR during split/organize:", e)
    raise

# [6/12] Data Augmentation Setup: Define augmentation pipeline
print("\n[6/12] Data Augmentation Setup - Defining augmentation pipeline (training only)")

# Image parameters
IMG_SIZE = (224, 224)
IMG_HEIGHT, IMG_WIDTH = IMG_SIZE
BATCH_SIZE = 16  # Optimal for T4 GPU as specified
LEARNING_RATE = 1e-4
MAX_EPOCHS = 50

# Augmentation pipeline for training
try:
    augmentation_layers = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.3),  # ±30% of 2π rad translated to layer expecting fraction of 1
        layers.RandomZoom(0.2),
        layers.RandomTranslation(0.2, 0.2),
        layers.RandomContrast(0.3),
        layers.RandomBrightness(0.2),
    ], name="data_augmentation")

    print("✓ Augmentation pipeline created")
except Exception as e:
    print("ERROR creating augmentation pipeline:", e)
    raise

# [7/12] Load Datasets: Create TensorFlow datasets from organized folders
print("\n[7/12] Load Datasets - Creating tf.data datasets from directories")

try:
    # Use image_dataset_from_directory for train/val/test (no internal split here)
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        train_dir,
        labels="inferred",
        label_mode="int",
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE,
        shuffle=True,
        seed=42
    )

    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        val_dir,
        labels="inferred",
        label_mode="int",
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE,
        shuffle=False
    )

    test_ds = tf.keras.preprocessing.image_dataset_from_directory(
        test_dir,
        labels="inferred",
        label_mode="int",
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE,
        shuffle=False
    )

    class_names = train_ds.class_names
    print("Class names (inferred):", class_names)
    print("✓ Datasets created via image_dataset_from_directory")
except Exception as e:
    print("ERROR creating datasets:", e)
    raise

# [8/12] Preprocessing Pipeline: Apply normalization and augmentation
print("\n[8/12] Preprocessing Pipeline - Applying normalization, augmentation, and performance optimizations")

# Normalization: scale pixel values to [0,1]
# Note: Since we enabled mixed precision, keep model output dtype handled appropriately.
rescale_layer = layers.Rescaling(1./255, name="rescale")

# Build final train/val/test pipelined datasets
AUTOTUNE = tf.data.AUTOTUNE

def prepare_for_training(dataset, augment=False):
    """
    Apply rescaling, optionally augmentation, cache, shuffle, batch, prefetch.
    """
    ds = dataset.map(lambda x, y: (rescale_layer(x), y), num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(lambda x, y: (augmentation_layers(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    if augment:
        ds = ds.shuffle(buffer_size=100, seed=42)
    ds = ds.prefetch(buffer_size=AUTOTUNE)
    return ds

train_ds = prepare_for_training(train_ds, augment=True)
val_ds = prepare_for_training(val_ds, augment=False)
test_ds = prepare_for_training(test_ds, augment=False)

print("✓ Preprocessing pipeline ready (augmentation applied to training only)")

# [9/12] Build VGG16 Model: Transfer learning model architecture
print("\n[9/12] Build VGG16 Model - Creating transfer learning model with frozen base")

try:
    # Load VGG16 base model without top classification layers
    base_model = tf.keras.applications.VGG16(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
    )
    base_model.trainable = False  # Freeze the base

    inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name="input_image")
    x = inputs
    # Base model expects inputs in range 0-255 preprocessed differently, but we used simple rescaling to [0,1].
    # Since we are using transfer learning and small dataset, this is acceptable per requirements.
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.Dropout(0.5, name="dropout_0")(x)
    x = layers.Dense(128, activation='relu', name="dense_128")(x)
    x = layers.Dropout(0.4, name="dropout_1")(x)
    # Final dense must output dtype float32 for numerical stability; when mixed precision policy is float16,
    # set dtype on the final layer to float32.
    outputs = layers.Dense(2, activation='softmax', dtype='float32', name="predictions")(x)

    model = Model(inputs, outputs, name="vgg16_transfer_ozone")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    model.summary(print_fn=lambda s: print(s))
    print("✓ Model built and compiled")
except Exception as e:
    print("ERROR building model:", e)
    raise

# [10/12] Setup Callbacks: Configure early stopping, checkpointing, LR reduction
print("\n[10/12] Setup Callbacks - EarlyStopping, ModelCheckpoint, ReduceLROnPlateau")

callbacks = []
try:
    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )

    checkpoint_path = "best_ozone_model.keras" # Changed to .keras
    model_checkpoint = ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )

    callbacks = [early_stopping, model_checkpoint, reduce_lr]
    print(f"✓ Callbacks configured: early_stopping (patience=5), checkpoint -> {checkpoint_path}, reduce_lr")
except Exception as e:
    print("ERROR setting callbacks:", e)
    raise

# [11/12] Train Model: Execute training with progress monitoring
print("\n[11/12] Train Model - Starting model.fit()")

try:
    history = model.fit(
        train_ds,
        epochs=MAX_EPOCHS,
        validation_data=val_ds,
        callbacks=callbacks,
        verbose=1
    )
    print("✓ Training completed")
except Exception as e:
    print("ERROR during training:", e)
    raise

# [12/12] Evaluate & Save: Test set evaluation, visualizations, model saving, inference
print("\n[12/12] Evaluate & Save - Evaluating on test set, creating plots, saving models")

try:
    # Evaluate on test set
    test_loss, test_acc = model.evaluate(test_ds, verbose=1)
    print(f"\nTest Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

    # Get predictions and true labels for confusion matrix
    y_true = []
    y_pred = []
    y_prob = []

    for batch_images, batch_labels in test_ds:
        preds = model.predict(batch_images)
        preds_labels = np.argmax(preds, axis=1)
        y_true.extend(batch_labels.numpy().tolist())
        y_pred.extend(preds_labels.tolist())
        y_prob.extend(np.max(preds, axis=1).tolist())

    # Confusion Matrix and Classification Report
    cm = confusion_matrix(y_true, y_pred)
    cls_report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    # Handle division by zero if a class has no true samples in the test set
    per_class_acc = np.diag(cm) / np.sum(cm, axis=1)
    per_class_acc = np.nan_to_num(per_class_acc) # Replace NaN with 0 where sum is 0

    print("\nConfusion Matrix:\n", cm)
    print("\nClassification Report:\n", cls_report)
    for idx, name in enumerate(class_names):
        print(f"Per-class accuracy for '{name}': {per_class_acc[idx]:.4f}")

    # Create and save visualizations
    os.makedirs("plots", exist_ok=True)

    # Training history plot (accuracy & loss)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Accuracy
    axes[0].plot(history.history.get('accuracy', []), label='train_accuracy')
    axes[0].plot(history.history.get('val_accuracy', []), label='val_accuracy')
    axes[0].set_title('Training vs Validation Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    # Loss
    axes[1].plot(history.history.get('loss', []), label='train_loss')
    axes[1].plot(history.history.get('val_loss', []), label='val_loss')
    axes[1].set_title('Training vs Validation Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    history_plot_path = os.path.join("plots", "training_history.png")
    plt.savefig(history_plot_path)
    plt.close(fig)
    print(f"Saved training history plot to {history_plot_path}")

    # Confusion matrix heatmap
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    cm_plot_path = os.path.join("plots", "confusion_matrix.png")
    plt.savefig(cm_plot_path)
    plt.close()
    print(f"Saved confusion matrix to {cm_plot_path}")

    # Accuracy comparison bar chart (train/val/test final accuracies)
    train_acc_final = history.history.get('accuracy', [])[-1] if history.history.get('accuracy') else None
    val_acc_final = history.history.get('val_accuracy', [])[-1] if history.history.get('val_accuracy') else None
    test_acc_final = test_acc

    plt.figure(figsize=(6, 5))
    labels = ['Train', 'Validation', 'Test']
    accs = [train_acc_final, val_acc_final, test_acc_final]
    sns.barplot(x=labels, y=accs)
    plt.ylim(0, 1)
    plt.ylabel('Accuracy')
    plt.title('Accuracy Comparison')
    acc_plot_path = os.path.join("plots", "accuracy_comparison.png")
    plt.savefig(acc_plot_path)
    plt.close()
    print(f"Saved accuracy comparison to {acc_plot_path}")

    # Save final models:
    final_keras_path = "ozone_vgg16_final.keras" # Changed to .keras
    saved_model_dir = "ozone_vgg16_savedmodel"
    # Save the best model (checkpoint) if exists else current model
    if os.path.exists(checkpoint_path):
        # Load checkpoint into a model object to ensure it's saved both ways
        try:
            best_model = tf.keras.models.load_model(checkpoint_path)
            best_model.save(final_keras_path) # Save best model in .keras format
            best_model.export(saved_model_dir) # Use export for SavedModel format
            print(f"Saved best model (from checkpoint) to {final_keras_path} and {saved_model_dir}/")
            model_to_report = best_model
        except Exception as e:
            print("Warning: Could not load checkpoint; saving current model instead:", e)
            model.save(final_keras_path) # Save current model in .keras format
            model.export(saved_model_dir) # Use export for SavedModel format
            print(f"Saved current model to {final_keras_path} and {saved_model_dir}/")
            model_to_report = model
    else:
        model.save(final_keras_path) # Save current model in .keras format
        model.export(saved_model_dir) # Use export for SavedModel format
        print(f"Saved current model to {final_keras_path} and {saved_model_dir}/")
        model_to_report = model

    # Demonstrate inference on a sample test image
    print("\nInference demonstration on a sample test image:")
    # Find a sample image path in test_dir
    sample_found = False
    for cls in ['normal', 'depleted']:
        cls_dir = os.path.join(test_dir, cls)
        if os.path.isdir(cls_dir) and os.listdir(cls_dir):
            sample_fname = os.listdir(cls_dir)[0]
            sample_path = os.path.join(cls_dir, sample_fname)
            sample_found = True
            sample_true_label = cls
            break

    if sample_found:
        # Load image
        img = tf.keras.preprocessing.image.load_img(sample_path, target_size=IMG_SIZE)
        img_arr = tf.keras.preprocessing.image.img_to_array(img)
        img_arr_rescaled = img_arr / 255.0
        input_tensor = np.expand_dims(img_arr_rescaled, axis=0)  # shape (1, H, W, 3)

        # Use the model_to_report (either best or current) for prediction
        preds = model_to_report.predict(input_tensor)
        pred_label_idx = int(np.argmax(preds, axis=1)[0])
        pred_label = class_names[pred_label_idx]
        confidence = float(np.max(preds)) * 100.0

        print(f"Sample image: {sample_fname}")
        print(f"True label: {sample_true_label}")
        print(f"Predicted label: {pred_label} with confidence {confidence:.2f}%")
    else:
        print("No sample image found in test set for inference demo.")

    # Print final summary
    print("\nFinal Summary:")
    print(f"- Number of images found and labeled: {len(df_images)}")
    print(f"- Class distribution: {class_counts}")
    print(f"- Train/Val/Test sizes: {len(train_df)}/{len(val_df)}/{len(test_df)}")
    print(f"- Test Accuracy: {test_acc:.4f}")
    print(f"- Saved model (.keras): {os.path.abspath(final_keras_path)}")
    print(f"- Saved model (SavedModel dir): {os.path.abspath(saved_model_dir)}")
    print(f"- Saved plots: {os.path.abspath('plots')}")

except Exception as e:
    print("ERROR during evaluation/save/inference:", e)
    raise

print("\nScript finished successfully. You can now download the saved models and plots from the Colab workspace.")

In [ ]:
"""
Improved & Easy-to-Understand Script:
Ozone Depletion Classification using VGG16 (transfer learning)

Key improvements:
- Clear step-by-step structure with small functions
- Robust file/path checks and helpful error messages
- Uses dataframe of image paths + labels (Excel or CSV)
- Image loading with normalization, optional augmentation (ImageDataGenerator)
- Class weights handling for imbalance
- Training, evaluation, confusion matrix, classification report, and plots
- Model saving (HDF5) and sample inference function

Usage:
- Edit DATA_FILE to point to your spreadsheet (Excel or CSV).
- Ensure image paths in the sheet are correct (absolute or relative).
- Run in Colab or local environment with required packages installed.

Dependencies:
pandas, numpy, matplotlib, seaborn, scikit-learn, tensorflow (2.x)
"""

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ----------------------------
# CONFIGURATION (edit as needed)
# ----------------------------
DATA_FILE = "/content/ozone_2017_2025_monthly_label.xlsx"  # or .csv
IMAGE_COL = "filename"   # column name with image file paths - Corrected from "image_path"
LABEL_COL = "label"        # column name with labels: "normal" / "depleted"
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-4
MODEL_SAVE_PATH = "ozone_vgg16_best.h5"
RANDOM_STATE = 42
IMAGES_PATH = '/content/uploaded_images/' # Define IMAGES_PATH here

# ----------------------------
# Helpful utility functions
# ----------------------------
def read_table(path):
    """Read an Excel or CSV into a DataFrame."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Label file not found: {path}")
    if p.suffix.lower() in [".xls", ".xlsx"]:
        return pd.read_excel(path)
    elif p.suffix.lower() == ".csv":
        return pd.read_csv(path)
    else:
        raise ValueError("Unsupported label file. Use .xlsx/.xls or .csv")

def validate_and_prepare_df(df):
    """Ensure image and label columns exist, drop missing files, normalize labels."""
    if IMAGE_COL not in df.columns or LABEL_COL not in df.columns:
        raise KeyError(f"DataFrame must contain columns '{IMAGE_COL}' and '{LABEL_COL}'")
    df = df[[IMAGE_COL, LABEL_COL]].copy()
    df = df.dropna()
    # Normalize label strings
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()
    # Keep only expected labels
    df = df[df[LABEL_COL].isin(["normal", "depleted"])]
    # Verify image paths exist; create absolute paths if necessary
    valid_rows = []
    missing_filenames = []
    for index, row in df.iterrows():
        fname = row[IMAGE_COL]
        # Construct the full path
        full_path = os.path.join(IMAGES_PATH, fname)
        if os.path.exists(full_path):
            # Store the full path in the DataFrame
            row[IMAGE_COL] = full_path
            valid_rows.append(row)
        else:
            missing_filenames.append(fname)

    if missing_filenames:
        print(f"Warning: {len(missing_filenames)} image paths not found in {IMAGES_PATH}. They will be removed:")
        for mf in missing_filenames:
            print(f" - {mf}")

    # Create a new DataFrame with only the valid rows (which now have full paths)
    df_validated = pd.DataFrame(valid_rows, columns=[IMAGE_COL, LABEL_COL])

    if df_validated.empty:
        raise RuntimeError(f"No valid image files found in {IMAGES_PATH}. Check the image paths in the label file and the {IMAGES_PATH} directory.")
    return df_validated


def load_and_preprocess_image(path, target_size=IMG_SIZE):
    """Load image, resize, convert to float32 and scale to [0,1]."""
    img = load_img(path, target_size=target_size)
    arr = img_to_array(img).astype("float32") / 255.0
    return arr

def build_model(img_size=IMG_SIZE, lr=LEARNING_RATE, dropout_rate=0.4):
    """Builds VGG16-based transfer learning model (base frozen)."""
    base = VGG16(weights="imagenet", include_top=False, input_shape=(img_size[0], img_size[1], 3))
    base.trainable = False  # freeze base
    inp = Input(shape=(img_size[0], img_size[1], 3))
    x = inp
    x = base(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    out = Dense(2, activation="softmax", dtype="float32")(x)
    model = Model(inp, out, name="vgg16_ozone")
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

def plot_history(history, out_dir="plots"):
    os.makedirs(out_dir, exist_ok=True)
    # Accuracy
    plt.figure(figsize=(8, 4))
    plt.plot(history.history.get("accuracy", []), label="train_acc")
    plt.plot(history.history.get("val_accuracy", []), label="val_acc")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    acc_path = os.path.join(out_dir, "accuracy.png")
    plt.savefig(acc_path)
    plt.close()
    # Loss
    plt.figure(figsize=(8, 4))
    plt.plot(history.history.get("loss", []), label="train_loss")
    plt.plot(history.history.get("val_loss", []), label="val_loss")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    loss_path = os.path.join(out_dir, "loss.png")
    plt.savefig(loss_path)
    plt.close()
    print(f"Saved training plots to '{out_dir}'")

def plot_confusion_matrix(y_true, y_pred, class_names, out_file="plots/confusion_matrix.png"):
    os.makedirs(os.path.dirname(out_file), exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.savefig(out_file)
    plt.close()
    print(f"Saved confusion matrix to '{out_file}'")

# ----------------------------
# MAIN PROCESS
# ----------------------------
def main():
    print("1) Loading labels...")
    df = read_table(DATA_FILE)
    print(f" - Original rows: {len(df)}")
    df = validate_and_prepare_df(df)
    print(f" - Valid rows with existing images: {len(df)}")
    print(df[LABEL_COL].value_counts().to_dict())

    # Map labels to integers
    label_map = {"normal": 0, "depleted": 1}
    df["label_id"] = df[LABEL_COL].map(label_map)

    # Train/val/test split (stratified)
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label_id"], random_state=RANDOM_STATE)
    val_df, test_df = train_test_split(temp_df, test_size=0.3333, stratify=temp_df["label_id"], random_state=RANDOM_STATE)
    # This yields approx: train 70%, val 20%, test 10%
    print(f"2) Split sizes -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    # Create ImageDataGenerators for augmentation (train) and simple rescaling (val/test)
    train_datagen = ImageDataGenerator(
        rescale=1.0/255.0,
        horizontal_flip=True,
        vertical_flip=True,
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.15,
        brightness_range=(0.8, 1.2)
    )
    val_test_datagen = ImageDataGenerator(rescale=1.0/255.0)

    # Use flow_from_dataframe to generate batches directly from paths
    # flow_from_dataframe expects filenames relative to a directory if 'directory' is set,
    # but providing full paths via x_col is supported by setting directory=None.
    train_gen = train_datagen.flow_from_dataframe(
        train_df,
        x_col=IMAGE_COL,
        y_col="label_id",
        target_size=IMG_SIZE,
        color_mode="rgb",
        class_mode="raw",      # returns labels as float array -> good for sparse_categorical_crossentropy
        batch_size=BATCH_SIZE,
        shuffle=True,
        directory=None # Ensure directory is None so x_col is treated as full path
    )
    val_gen = val_test_datagen.flow_from_dataframe(
        val_df,
        x_col=IMAGE_COL,
        y_col="label_id",
        target_size=IMG_SIZE,
        color_mode="rgb",
        class_mode="raw",
        batch_size=BATCH_SIZE,
        shuffle=False,
        directory=None # Ensure directory is None so x_col is treated as full path
    )
    test_gen = val_test_datagen.flow_from_dataframe(
        test_df,
        x_col=IMAGE_COL,
        y_col="label_id",
        target_size=IMG_SIZE,
        color_mode="rgb",
        class_mode="raw",
        batch_size=BATCH_SIZE,
        shuffle=False,
        directory=None # Ensure directory is None so x_col is treated as full path
    )

    # Compute class weights from training labels to handle imbalance
    classes = np.unique(train_df["label_id"])
    class_weights_values = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["label_id"].values)
    class_weights = {int(c): w for c, w in zip(classes, class_weights_values)}
    print(f"3) Computed class weights: {class_weights}")

    # Build model
    print("4) Building model...")
    model = build_model(img_size=IMG_SIZE, lr=LEARNING_RATE)
    model.summary()

    # Callbacks
    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        ModelCheckpoint(MODEL_SAVE_PATH, monitor="val_accuracy", save_best_only=True, verbose=1)
    ]

    # Steps per epoch (use ceil to ensure all samples seen)
    steps_per_epoch = int(np.ceil(len(train_gen) ))
    validation_steps = int(np.ceil(len(val_gen) ))

    # Train
    print("5) Training...")
    history = model.fit(
        train_gen,
        epochs=EPOCHS,
        validation_data=val_gen,
        class_weight=class_weights,
        callbacks=callbacks,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        verbose=1
    )

    # Evaluation on test set
    print("6) Evaluating on test set...")
    # Predict probabilities then labels
    test_steps = int(np.ceil(len(test_gen)))
    y_true = test_df["label_id"].values
    # Reset test_gen to ensure consistent ordering
    test_gen.reset()
    preds_prob = model.predict(test_gen, steps=test_steps, verbose=1)
    y_pred = np.argmax(preds_prob, axis=1)

    acc = accuracy_score(y_true, y_pred)
    print(f" - Test Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Depleted"]))

    # Confusion matrix and plots
    plot_confusion_matrix(y_true, y_pred, class_names=["Normal", "Depleted"])
    plot_history(history, out_dir="plots")

    # Save best model path printed by callback
    if os.path.exists(MODEL_SAVE_PATH):
        print(f"Model saved to: {MODEL_SAVE_PATH}")
    else:
        # Save current model if checkpoint failed
        model.save(MODEL_SAVE_PATH)
        print(f"Model saved to: {MODEL_SAVE_PATH}")

    # Sample inference function demonstration (predict on single image path)
    def predict_single(image_path, model, label_map_inv={0: "normal", 1: "depleted"}):
        arr = load_and_preprocess_image(image_path, target_size=IMG_SIZE)
        arr = np.expand_dims(arr, axis=0)
        prob = model.predict(arr)[0]
        idx = int(np.argmax(prob))
        return label_map_inv[idx], float(np.max(prob))

    # Show a few random test predictions
    print("\n7) Sample predictions on test images:")
    sample_paths = test_df[IMAGE_COL].sample(n=min(5, len(test_df)), random_state=RANDOM_STATE).tolist()
    for p in sample_paths:
        # The paths in test_df[IMAGE_COL] are now full paths
        full_image_path = p
        if os.path.exists(full_image_path):
            pred_label, conf = predict_single(full_image_path, model)
            # Need to find the row in the original df or test_df using the full path
            # Finding the true label by full path
            true_label_row = df[df[IMAGE_COL] == full_image_path].iloc[0]
            true_label = true_label_row[LABEL_COL]
            print(f" - {Path(full_image_path).name}: Predicted = {pred_label} ({conf*100:.1f}%), True = {true_label}")
        else:
            print(f" - Image file not found for sample prediction: {full_image_path}")


    print("\nDone.")

if __name__ == "__main__":
    main()

In [ ]:
"""
Ozone Depletion Classification (end-to-end)
- Uses VGG16 transfer learning (frozen base)
- Reads labels CSV/Excel with columns: 'filename' and 'label' ('normal' or 'depleted')
- Expects image files in IMAGES_PATH (filenames in CSV matched to files in folder)
- Splits data (train/val/test ≈ 70/20/10), uses augmentation, computes class weights
- Trains model, evaluates on test set, saves model and plots (accuracy/loss, confusion matrix)
- Includes improved plotting utilities with smoothing and best-epoch annotations

Usage:
- Edit DATA_FILE and IMAGES_PATH as needed.
- Run in Colab or local environment with dependencies installed:
  pandas, numpy, matplotlib, seaborn, scikit-learn, tensorflow (2.x)
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ----------------------------
# CONFIGURATION (edit as needed)
# ----------------------------
DATA_FILE = "/content/ozone_2017_2025_monthly_label.xlsx"  # or .csv
IMAGE_COL = "filename"   # column in CSV with image filename (e.g., "2019-09-15.png")
LABEL_COL = "label"      # column with 'normal' / 'depleted'
IMAGES_PATH = "/content/uploaded_images"  # folder where images are stored
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-4
MODEL_SAVE_PATH = "ozone_vgg16_best.h5"
PLOTS_DIR = "plots"
RANDOM_STATE = 42
SMOOTH_ALPHA = 0.6  # smoothing for plots (0 = off)

# ----------------------------
# Utility & plotting functions
# ----------------------------
def read_table(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Label file not found: {path}")
    if p.suffix.lower() in [".xls", ".xlsx"]:
        return pd.read_excel(path)
    elif p.suffix.lower() == ".csv":
        return pd.read_csv(path)
    else:
        raise ValueError("Unsupported label file. Use .xlsx/.xls or .csv")

def validate_and_prepare_df(df):
    if IMAGE_COL not in df.columns or LABEL_COL not in df.columns:
        raise KeyError(f"Label file must include columns '{IMAGE_COL}' and '{LABEL_COL}'")
    df = df[[IMAGE_COL, LABEL_COL]].copy()
    df = df.dropna()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()
    df = df[df[LABEL_COL].isin(["normal", "depleted"])].reset_index(drop=True)

    # Get list of actual image files in the directory
    uploaded_image_files = {f for f in os.listdir(IMAGES_PATH) if f.lower().endswith('.png')}

    valid_rows = []
    missing_filenames = []
    for _, row in df.iterrows():
        fname = str(row[IMAGE_COL]).strip()
        if fname in uploaded_image_files:
             # Store the full path in the DataFrame
            full_path = os.path.join(IMAGES_PATH, fname)
            valid_rows.append({"filename": full_path, "label": row[LABEL_COL]})
        else:
            missing_filenames.append(fname)


    if missing_filenames:
        print(f"Warning: {len(missing_filenames)} files listed in label file not found in {IMAGES_PATH}. They will be ignored.")
        # Optionally list them; comment out if many:
        for m in missing_filenames[:20]:
            print("  -", m)
        if len(missing_filenames) > 20:
            print(f"  ... and {len(missing_filenames)-20} more")

    df_valid = pd.DataFrame(valid_rows)
    if df_valid.empty:
        raise RuntimeError(f"No valid images found in {IMAGES_PATH}. Check filenames in {DATA_FILE}.")
    return df_valid

def load_and_preprocess_image(path, target_size=IMG_SIZE):
    img = load_img(path, target_size=target_size)
    arr = img_to_array(img).astype("float32") / 255.0
    return arr

def build_model(img_size=IMG_SIZE, lr=LEARNING_RATE, dropout_rate=0.4):
    base = VGG16(weights="imagenet", include_top=False, input_shape=(img_size[0], img_size[1], 3))
    base.trainable = False
    inp = Input(shape=(img_size[0], img_size[1], 3))
    x = base(inp, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    out = Dense(2, activation="softmax", dtype="float32")(x)
    model = Model(inp, out, name="vgg16_ozone")
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

# Plotting utilities (smoothing, combined & separate plots)
def _smooth(values, alpha=SMOOTH_ALPHA):
    if values is None:
        return None
    if alpha <= 0:
        return np.array(values)
    out = []
    for v in values:
        if not out:
            out.append(v)
        else:
            out.append(alpha * out[-1] + (1 - alpha) * v)
    return np.array(out)

def _best_epoch_info(history, metric_key, higher_is_better=True):
    vals = history.history.get(metric_key)
    if vals is None:
        return None, None
    idx = int(np.argmax(vals)) if higher_is_better else int(np.argmin(vals))
    return idx + 1, vals[idx]

def plot_training_summary(history, out_dir=PLOTS_DIR, smooth=True, show=False, dpi=150):
    os.makedirs(out_dir, exist_ok=True)
    # extract
    train_acc = history.history.get("accuracy") or history.history.get("acc")
    val_acc = history.history.get("val_accuracy") or history.history.get("val_acc")
    train_loss = history.history.get("loss")
    val_loss = history.history.get("val_loss")
    epochs = np.arange(1, len(train_loss) + 1)

    if smooth:
        ta = _smooth(train_acc)
        va = _smooth(val_acc)
        tl = _smooth(train_loss)
        vl = _smooth(val_loss)
    else:
        ta, va, tl, vl = np.array(train_acc), np.array(val_acc), np.array(train_loss), np.array(val_loss)

    best_val_acc_epoch, best_val_acc = _best_epoch_info(history, "val_accuracy", higher_is_better=True)
    best_val_loss_epoch, best_val_loss = _best_epoch_info(history, "val_loss", higher_is_better=False)

    # Separate accuracy
    plt.figure(figsize=(9, 4))
    plt.plot(epochs, ta, label="Train Accuracy", marker="o", linewidth=1)
    plt.plot(epochs, va, label="Val Accuracy", marker="o", linewidth=1)
    if best_val_acc_epoch:
        plt.axvline(best_val_acc_epoch, color="green", linestyle="--", alpha=0.6)
        plt.annotate(f"best val acc\nepoch {best_val_acc_epoch}\n{best_val_acc:.3f}",
                     xy=(best_val_acc_epoch, best_val_acc),
                     xytext=(best_val_acc_epoch + 0.5, max(ta.min(), va.min()) + 0.02),
                     arrowprops=dict(arrowstyle="->", color="green"), fontsize=9, color="green")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.grid(alpha=0.3)
    plt.legend()
    acc_path = os.path.join(out_dir, "accuracy.png")
    plt.tight_layout()
    plt.savefig(acc_path, dpi=dpi)
    if show: plt.show()
    plt.close()

    # Separate loss
    plt.figure(figsize=(9, 4))
    plt.plot(epochs, tl, label="Train Loss", marker="o", linewidth=1)
    plt.plot(epochs, vl, label="Val Loss", marker="o", linewidth=1)
    if best_val_loss_epoch:
        plt.axvline(best_val_loss_epoch, color="red", linestyle="--", alpha=0.6)
        plt.annotate(f"best val loss\nepoch {best_val_loss_epoch}\n{best_val_loss:.3f}",
                     xy=(best_val_loss_epoch, best_val_loss),
                     xytext=(best_val_loss_epoch + 0.5, max(tl.max(), vl.max()) * 0.9),
                     arrowprops=dict(arrowstyle="->", color="red"), fontsize=9, color="red")
    ax2.set_title("Loss")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.grid(alpha=0.3)
    ax2.legend()

    combined_path = os.path.join(out_dir, "training_summary.png")
    plt.tight_layout()
    plt.savefig(combined_path, dpi=dpi)
    if show: plt.show()
    plt.close()

    print(f"Saved training plots: {acc_path}, {loss_path}, {combined_path}")

def plot_confusion_matrix(y_true, y_pred, class_names, out_file=os.path.join(PLOTS_DIR, "confusion_matrix.png")):
    os.makedirs(os.path.dirname(out_file), exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(out_file, dpi=150)
    plt.close()
    print(f"Saved confusion matrix: {out_file}")

# ----------------------------
# Main flow
# ----------------------------
def main():
    print("1) Loading label file...")
    df_raw = read_table(DATA_FILE)
    print(f" - Rows in label file: {len(df_raw)}")

    print("2) Validating image paths and labels...")
    df = validate_and_prepare_df(df_raw)
    print(f" - Valid image rows: {len(df)}")
    print(" - Class distribution:", df[LABEL_COL].value_counts().to_dict())

    # map labels to ints
    label_map = {"normal": 0, "depleted": 1}
    df["label_id"] = df[LABEL_COL].map(label_map)

    # split (approx 70/20/10)
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label_id"], random_state=RANDOM_STATE)
    val_df, test_df = train_test_split(temp_df, test_size=0.3333, stratify=temp_df["label_id"], random_state=RANDOM_STATE)
    print(f"3) Split sizes -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    # data generators
    train_datagen = ImageDataGenerator(
        rescale=1.0/255.0,
        horizontal_flip=True,
        vertical_flip=True,
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.15,
        brightness_range=(0.8, 1.2)
    )
    val_test_datagen = ImageDataGenerator(rescale=1.0/255.0)

    train_gen = train_datagen.flow_from_dataframe(
        train_df,
        x_col="filename",
        y_col="label_id",
        target_size=IMG_SIZE,
        color_mode="rgb",
        class_mode="raw",
        batch_size=BATCH_SIZE,
        shuffle=True,
        directory=None # Ensure directory is None so x_col is treated as full path
    )
    val_gen = val_test_datagen.flow_from_dataframe(
        val_df,
        x_col="filename",
        y_col="label_id",
        target_size=IMG_SIZE,
        color_mode="rgb",
        class_mode="raw",
        batch_size=BATCH_SIZE,
        shuffle=False,
        directory=None # Ensure directory is None so x_col is treated as full path
    )
    test_gen = val_test_datagen.flow_from_dataframe(
        test_df,
        x_col="filename",
        y_col="label_id",
        target_size=IMG_SIZE,
        color_mode="rgb",
        class_mode="raw",
        batch_size=BATCH_SIZE,
        shuffle=False,
        directory=None # Ensure directory is None so x_col is treated as full path
    )

    # class weights
    classes = np.unique(train_df["label_id"])
    cw_values = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["label_id"].values)
    class_weights = {int(c): float(w) for c, w in zip(classes, cw_weights_values)}
    print("4) Class weights:", class_weights)

    # model
    print("5) Building model...")
    model = build_model(img_size=IMG_SIZE, lr=LEARNING_RATE)
    model.summary()

    # callbacks
    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        ModelCheckpoint(MODEL_SAVE_PATH, monitor="val_accuracy", save_best_only=True, verbose=1)
    ]

    # compute steps
    steps_per_epoch = int(np.ceil(len(train_gen)))
    validation_steps = int(np.ceil(len(val_gen)))

    # train
    print("6) Training...")
    history = model.fit(
        train_gen,
        epochs=EPOCHS,
        validation_data=val_gen,
        class_weight=class_weights,
        callbacks=callbacks,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        verbose=1
    )

    # plots
    print("7) Creating training plots...")
    plot_training_summary(history, out_dir=PLOTS_DIR, smooth=(SMOOTH_ALPHA > 0), show=False)

    # evaluation
    print("8) Evaluating on test set...")
    test_steps = int(np.ceil(len(test_gen)))
    test_gen.reset()
    preds_prob = model.predict(test_gen, steps=test_steps, verbose=1)
    y_pred = np.argmax(preds_prob, axis=1)
    y_true = test_df["label_id"].values
    acc = accuracy_score(y_true, y_pred)
    print(f" - Test Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Depleted"], digits=4))

    plot_confusion_matrix(y_true, y_pred, class_names=["Normal", "Depleted"], out_file=os.path.join(PLOTS_DIR, "confusion_matrix.png"))

    # save model (ModelCheckpoint should have saved best already)
    if os.path.exists(MODEL_SAVE_PATH):
        print(f"Model saved to: {MODEL_SAVE_PATH}")
    else:
        model.save(MODEL_SAVE_PATH)
        print(f"Model saved to: {MODEL_SAVE_PATH}")

    # sample predictions
    print("9) Sample predictions:")
    def predict_single(path):
        arr = load_and_preprocess_image(path, target_size=IMG_SIZE)
        arr = np.expand_dims(arr, axis=0)
        prob = model.predict(arr)[0]
        idx = int(np.argmax(prob))
        return {0: "normal", 1: "depleted"}[idx], float(np.max(prob))

    sample_paths = test_df["filename"].sample(n=min(5, len(test_df)), random_state=RANDOM_STATE).tolist()
    for p in sample_paths:
        # The paths in test_df["filename"] are now full paths from validate_and_prepare_df
        full_image_path = p
        if os.path.exists(full_image_path):
            pred_label, conf = predict_single(full_image_path, model)
            true_label = test_df[test_df["filename"] == full_image_path][LABEL_COL].values[0]
            print(f" - {Path(full_image_path).name}: Predicted={pred_label} ({conf*100:.1f}%), True={true_label}")
        else:
            print(" - Sample file missing:", p)

    print("\nFinished. Plots saved to:", os.path.abspath(PLOTS_DIR))

if __name__ == "__main__":
    main()